# Dual-SR830 frequency and current–voltage sweep browser

Read-only analysis for standalone frequency and excitation sweep JSON/JSONL files. Each scan renders six separate twin-axis figures: Vxx and Vxy for harmonic orders h1, h2, and h3. The left axis is SR830 R (voltage magnitude) and the right axis is measured phase. Missing harmonic data is labeled explicitly; no values are inferred.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from attodry_control.commissioning_analysis import (
    ExcitationPathResistance,
    discover_commissioning_records,
    excitation_path_from_sweep_files,
    export_commissioning_csv,
    load_sweep_sample_files,
    plot_six_role_harmonic_sweeps,
)

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory.parent
    if working_directory.name.lower() == 'notebooks'
    else working_directory
)
DATA_DIRECTORY = PROJECT_ROOT / 'run_data' / 'commissioning'
DATA_DIRECTORY

## Start here: filters, remote record selection, and current calibration

Set `DATA_DIRECTORY` once in the preceding cell. Click **Refresh records**, choose one frequency and one excitation record from the remote-directory lists, then click **Load selected records**. The `completed` checkbox is on by default; deselect it only for an explicit audit. Current is always `SINE OUT RMS voltage / (external series + SR830 output + approximate device resistance)`, using the path archived with each sweep from `hardware.local.toml`. Do not duplicate normal resistance values here.

In [ ]:
completed_only_widget = widgets.Checkbox(
    value=True,
    description='Only completed records',
)
sample_status_widget = widgets.SelectMultiple(
    options=('clean', 'problem', 'unlocked', 'overload', 'instrument_error'),
    value=('clean',),
    description='Formal samples',
)
include_rejected_widget = widgets.Checkbox(
    value=False,
    description='Allow rejected audit records',
)
refresh_records_button = widgets.Button(
    description='Refresh records',
    icon='refresh',
    button_style='info',
)
frequency_record_widget = widgets.Dropdown(
    options=(),
    description='Frequency',
    layout=widgets.Layout(width='95%'),
)
excitation_record_widget = widgets.Dropdown(
    options=(),
    description='Excitation',
    layout=widgets.Layout(width='95%'),
)
load_selected_records_button = widgets.Button(
    description='Load selected records',
    icon='check',
    button_style='success',
)
selector_message = widgets.HTML()

FREQUENCY_PATHS = ()
EXCITATION_PATHS = ()

def _sync_filters():
    global RECORD_STATUSES, SAMPLE_STATUSES, INCLUDE_REJECTED
    RECORD_STATUSES = {'completed'} if completed_only_widget.value else None
    SAMPLE_STATUSES = set(sample_status_widget.value)
    INCLUDE_REJECTED = include_rejected_widget.value

def _record_options(scan_type):
    records = discover_commissioning_records(
        DATA_DIRECTORY,
        record_statuses=RECORD_STATUSES,
        scan_types={scan_type},
    )
    return [
        (
            f'{record.path.name} | {record.record_status} | '
            f'{record.sample_count} formal samples',
            str(record.path),
        )
        for record in records
    ]

def _refresh_records(_=None):
    _sync_filters()
    frequency_options = _record_options('frequency')
    excitation_options = _record_options('excitation')
    frequency_record_widget.options = frequency_options
    excitation_record_widget.options = excitation_options
    selector_message.value = (
        f'<b>Directory:</b> {DATA_DIRECTORY}<br>'
        f'Found {len(frequency_options)} frequency and '
        f'{len(excitation_options)} excitation records.'
    )

def _load_selected_records(_):
    global FREQUENCY_PATHS, EXCITATION_PATHS
    _sync_filters()
    frequency_path = frequency_record_widget.value
    excitation_path = excitation_record_widget.value
    if not frequency_path or not excitation_path:
        selector_message.value = (
            '<b>No matching completed pair:</b> refresh the lists or adjust '
            'the record-status filters.'
        )
        return
    FREQUENCY_PATHS = (Path(frequency_path),)
    EXCITATION_PATHS = (Path(excitation_path),)
    selector_message.value = (
        '<b>Selected records loaded.</b> Run the catalog and plot cells below '
        'to refresh the figures.'
    )

refresh_records_button.on_click(_refresh_records)
load_selected_records_button.on_click(_load_selected_records)
_refresh_records()
display(widgets.VBox([
    widgets.HBox([refresh_records_button, load_selected_records_button, completed_only_widget, include_rejected_widget]),
    sample_status_widget,
    frequency_record_widget,
    excitation_record_widget,
    selector_message,
]))

_sync_filters()

# Default: use measurement_config.excitation_path recorded in each selected JSON.
# Set this only for legacy JSON that lacks that snapshot; it deliberately
# overrides every selected file, so do not use it for normal daily records.
EXCITATION_PATH_OVERRIDE: ExcitationPathResistance | None = None
# EXCITATION_PATH_OVERRIDE = ExcitationPathResistance(
#     external_series_resistance_ohm=...,
#     sr830_output_resistance_ohm=...,
#     approximate_device_resistance_ohm=...,
# )
# Plot phase only above this amplitude and below this circular sample spread.
# Set the amplitude to 0.0 and the spread to None to inspect all raw phases.
PHASE_MINIMUM_AMPLITUDE_V = 1e-6
PHASE_MAXIMUM_STANDARD_DEVIATION_DEG = 5.0
# The selection UI above sets these tuples. Leave the default fallback below
# unchanged when the newest completed record of each type is desired.

{
    'current_calibration': (
        'recorded measurement_config.excitation_path'
        if EXCITATION_PATH_OVERRIDE is None
        else 'explicit legacy override'
    ),
    'filters': {
        'record_statuses': RECORD_STATUSES,
        'sample_statuses': SAMPLE_STATUSES,
        'include_rejected': INCLUDE_REJECTED,
    },
    'phase_display': {
        'minimum_amplitude_v': PHASE_MINIMUM_AMPLITUDE_V,
        'maximum_standard_deviation_deg': PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
    },
}

## Filtered catalog

The catalog is newest-first. Toggle the completed checkbox, choose formal-sample statuses, then click **Refresh records**. The two remote-directory selectors replace the frequency and excitation defaults after **Load selected records**.

In [ ]:
RECORD_STATUSES = {'completed'} if completed_only_widget.value else None
SAMPLE_STATUSES = set(sample_status_widget.value)
INCLUDE_REJECTED = include_rejected_widget.value

catalog = discover_commissioning_records(
    DATA_DIRECTORY,
    record_statuses=RECORD_STATUSES,
    scan_types={'frequency', 'excitation'},
)
[
    {
        'file': item.path.name,
        'scan': item.scan_type,
        'status': item.record_status,
        'samples': item.sample_count,
        'problem_samples': item.problem_count,
        'error': item.error,
    }
    for item in catalog
]

## Load selected formal samples

The loader excludes transition and cleanup payloads. It refuses rejected records unless `INCLUDE_REJECTED=True`. Frequency and excitation records stay separate.

In [ ]:
completed_catalog = discover_commissioning_records(
    DATA_DIRECTORY,
    record_statuses={'completed'},
    scan_types={'frequency', 'excitation'},
)
frequency_record = next(
    (item for item in completed_catalog if item.scan_type == 'frequency'), None
)
excitation_record = next(
    (item for item in completed_catalog if item.scan_type == 'excitation'), None
)
if not FREQUENCY_PATHS and frequency_record is None:
    raise FileNotFoundError('A completed frequency record is missing.')
if not EXCITATION_PATHS and excitation_record is None:
    raise FileNotFoundError('A completed excitation record is missing.')

frequency_paths = tuple(FREQUENCY_PATHS) or (frequency_record.path,)
excitation_paths = tuple(EXCITATION_PATHS) or (excitation_record.path,)
frequency_rows = load_sweep_sample_files(
    frequency_paths,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
)
excitation_rows = load_sweep_sample_files(
    excitation_paths,
    include_rejected=INCLUDE_REJECTED,
    sample_statuses=SAMPLE_STATUSES,
)
frequency_excitation_path = excitation_path_from_sweep_files(
    frequency_paths,
    excitation_path_override=EXCITATION_PATH_OVERRIDE,
)
excitation_excitation_path = excitation_path_from_sweep_files(
    excitation_paths,
    excitation_path_override=EXCITATION_PATH_OVERRIDE,
)
{
    'frequency_files': frequency_paths,
    'frequency_samples': len(frequency_rows),
    'excitation_files': excitation_paths,
    'excitation_samples': len(excitation_rows),
    'frequency_total_path_resistance_ohm': frequency_excitation_path.total_resistance_ohm,
    'excitation_total_path_resistance_ohm': excitation_excitation_path.total_resistance_ohm,
}

## Six frequency figures and six current–voltage figures

Frequency figures use a logarithmic frequency axis and state the SINE OUT-derived RMS current in their titles. Current–voltage figures use the same calculated current on the x axis. Each figure keeps voltage magnitude and phase on separate y axes.

In [ ]:
frequency_figures = plot_six_role_harmonic_sweeps(
    frequency_rows,
    excitation_path=frequency_excitation_path,
    phase_minimum_amplitude_v=PHASE_MINIMUM_AMPLITUDE_V,
    phase_maximum_standard_deviation_deg=PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
)
current_voltage_figures = plot_six_role_harmonic_sweeps(
    excitation_rows,
    excitation_path=excitation_excitation_path,
    phase_minimum_amplitude_v=PHASE_MINIMUM_AMPLITUDE_V,
    phase_maximum_standard_deviation_deg=PHASE_MAXIMUM_STANDARD_DEVIATION_DEG,
)

for scan_name, figures in (
    ('frequency', frequency_figures),
    ('current_voltage', current_voltage_figures),
):
    print(f'{scan_name}: {len(figures)} figures')
    for (role, harmonic), figure in figures.items():
        display(figure)
        plt.close(figure)

## Optional export

No files are written unless `SAVE_OUTPUTS=True`. Exports are placed under the ignored analysis-output directory.

In [ ]:
SAVE_OUTPUTS = False
OUTPUT_DIRECTORY = PROJECT_ROOT / 'analysis_output' / 'sr830_commissioning'
if SAVE_OUTPUTS:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    export_commissioning_csv(
        frequency_rows, OUTPUT_DIRECTORY / 'frequency_samples.csv'
    )
    export_commissioning_csv(
        excitation_rows, OUTPUT_DIRECTORY / 'excitation_samples.csv'
    )
    for scan_name, figures in (
        ('frequency', frequency_figures),
        ('current_voltage', current_voltage_figures),
    ):
        for (role, harmonic), figure in figures.items():
            stem = f'{scan_name}_{role}_h{harmonic}'
            figure.savefig(OUTPUT_DIRECTORY / f'{stem}.png', dpi=200)
            figure.savefig(OUTPUT_DIRECTORY / f'{stem}.pdf')
    display(OUTPUT_DIRECTORY)